### Dataset Architecture & Feature Specification

A **hybrid dataset** integrating **119,404 real-world physical records** from Kaggle's *Clothes Size Prediction* dataset, complemented by synthetic garment parameters and target sizing logic.

#### 1. Data Generation & Partitioning
* **Data Origin:** Real human physical dimensions (`height`, `weight`) extracted directly from Kaggle's `final_test.csv` ($N = 119,404$ records).
* **Complemented Features:** Synthetic garment specifications (`garment_chest_cm`, `fabric_stretch_pct`, `product_type_id`) generated to represent retail inventory.
* **Sampling & Partitioning:** To ensure fast optimization during Grid Search, a stratified subset of 15,000 records was sampled and split:
  * **Training Set:** 12,000 samples (80%)
  * **Test Set:** 3,000 samples (20%)
* **Target Classes:** Fit outcomes derived from chest-to-garment physical dimension differentials:
  * `0`: **Too Small** (Garment chest smaller than estimated body chest)
  * `1`: **Good Fit** (Garment chest matches body chest within $\pm 4.0\text{ cm}$)
  * `2`: **Too Large** (Garment chest larger than estimated body chest)

#### 2. Feature Specifications & Physical Constraints

| Feature Name | Data Type | Data Source / Distribution | Physical Bounds | Description |
| :--- | :--- | :--- | :--- | :--- |
| **`height_cm`** | Continuous | Kaggle real data (`height` column) | 140.0 cm to 220.0 cm | User physical height. |
| **`weight_kg`** | Continuous | Kaggle real data (`weight` column) | 40.0 kg to 150.0 kg | User physical weight. |
| **`imc_index`** | Continuous | Calculated feature | Dynamic (10.0 to 50.0+) | Derived Body Mass Index (BMI). |
| **`garment_chest_cm`** | Continuous | Synthesized ($\mu=95.0, \sigma=15.0$) | 60.0 cm to 140.0 cm | Flat chest width of garment. |
| **`fabric_stretch_pct`** | Continuous | Synthesized ($\mu=5.0, \sigma=5.0$) | 0.0% to 20.0% | Elasticity factor of material. |
| **`product_type_id`** | Categorical | Discrete integers (`0`, `1`, `2`) | `0`, `1`, `2` | Garment category (0: Tops, 1: Bottoms, 2: Jackets). |

#### 3. Data Transformations & Feature Engineering
* **Derived BMI (`imc_index`):** Computed dynamically using body dimensions:
  $$\text{imc\_index} = \frac{\text{weight\_kg}}{\left(\frac{\text{height\_cm}}{100}\right)^2}$$

* **Heuristic Target Labeling:** Target `fit_class` labels are created using estimated chest clearance:
  $$\text{estimated\_body\_chest} = 0.45 \times \text{weight\_kg} + 0.35 \times \text{height\_cm}$$
  $$\text{chest\_delta} = (\text{garment\_chest\_cm} + 0.5 \times \text{fabric\_stretch\_pct}) - \text{estimated\_body\_chest}$$

* **One-Hot Encoding:** `product_type_id` is encoded via `pd.get_dummies()`, expanding the schema to **8 feature columns** stored in `model_features.joblib`:
  1. `height_cm`
  2. `weight_kg`
  3. `imc_index`
  4. `garment_chest_cm`
  5. `fabric_stretch_pct`
  6. `product_type_id_0`
  7. `product_type_id_1`
  8. `product_type_id_2`

### 0. Environment & MLflow Tracking Setup

This notebook runs inside the `fitcheck-jupyter` container (`docker compose up -d --build`, then open http://localhost:8888).
All dependencies are installed in the image, so no `!pip install` is needed.

* **MLflow UI / tracking server:** http://localhost:5001 (from inside the container it is reached as `http://mlflow:5001`).
* **Models folder:** `Models/`. The API (`fitcheck-api`, http://localhost:3000) serves the `.joblib` files saved here.
* **Run this cell first:** every later cell logs to the tracking server configured below.

In [ ]:
import os
import sys
from pathlib import Path

import mlflow

# ------------------------------------------------------------------
# Project paths (the notebook runs from /app/Notebooks inside Docker)
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
MODEL_DIR = Path(os.getenv("MODEL_DIR", PROJECT_ROOT / "Models"))   # read by the API
SRC_DIR = PROJECT_ROOT / "src"                                       # shared feature code
DATA_PATH = PROJECT_ROOT / "data" / "final_test.csv"                 # cached Kaggle CSV
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Make `fitcheck` (src/) and `api` importable, same as the API container does via PYTHONPATH
for p in (SRC_DIR, PROJECT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ------------------------------------------------------------------
# MLflow: log to the tracking server instead of a local ./mlruns folder
# ------------------------------------------------------------------
# In docker compose this env var is http://mlflow:5001; outside Docker fall back to localhost.
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# A new experiment name is used on purpose: the old "FitCheck_Sizing_Model" experiment was
# created in Colab and stores artifacts under /content/mlruns, a path that does not exist here.
# New experiments get the server's artifact store (mlflow-artifacts:/ -> ./artifacts on disk).
EXPERIMENT_NAME = "FitCheck_Sizing_Model_Docker"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("Experiment          :", experiment.name, "| artifacts ->", experiment.artifact_location)
print("Models folder       :", MODEL_DIR)

In [ ]:
import kagglehub
import os
import shutil

# kagglehub is already installed in the Docker image (no !pip install needed).
# The CSV is cached in data/ (mounted from your machine) so it is only downloaded once.
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if DATA_PATH.exists():
    print(f"Using cached dataset: {DATA_PATH}")
else:
    # Download dataset from Kaggle
    path = kagglehub.dataset_download("tourist55/clothessizeprediction")

    # Copy the downloaded CSV into data/ as final_test.csv
    for file in os.listdir(path):
        if file.endswith(".csv"):
            shutil.copy(os.path.join(path, file), DATA_PATH)
            print(f"Loaded {file} as {DATA_PATH}")

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# TASK 1: HYBRID DATASET (Kaggle Real Data + Complemented Features)
# ==========================================

# 1. Load Kaggle dataset (downloaded to data/final_test.csv by the previous cell)
kaggle_raw = pd.read_csv(DATA_PATH).dropna(subset=["height", "weight"])

np.random.seed(42)
n_samples = len(kaggle_raw)

# 2. Extract real physical human dimensions
height_cm = np.clip(kaggle_raw["height"].values, 140.0, 220.0)
weight_kg = np.clip(kaggle_raw["weight"].values, 40.0, 150.0)

# 3. Calculate derived BMI (imc_index)
imc_index = weight_kg / ((height_cm / 100.0) ** 2)

# 4. Complement missing garment & apparel features
garment_chest_cm = np.clip(np.random.normal(95.0, 15.0, size=n_samples), 60.0, 140.0)
fabric_stretch_pct = np.clip(np.random.normal(5.0, 5.0, size=n_samples), 0.0, 20.0)
product_type_id = np.random.choice([0, 1, 2], size=n_samples)

# 5. Generate realistic fit_class target based on body vs garment chest differential
estimated_body_chest = 0.45 * weight_kg + 0.35 * height_cm
chest_delta = (garment_chest_cm + (fabric_stretch_pct * 0.5)) - estimated_body_chest

# Target labels: 0 = Too Small, 1 = Good Fit, 2 = Too Large
fit_class = np.where(chest_delta < -4.0, 0, np.where(chest_delta > 4.0, 2, 1))

# 6. Define schema matching downstream ML pipeline
feature_names = [
    'height_cm', 'weight_kg', 'imc_index',
    'garment_chest_cm', 'fabric_stretch_pct', 'product_type_id'
]

fitcheck_df = pd.DataFrame({
    'height_cm': height_cm,
    'weight_kg': weight_kg,
    'imc_index': imc_index,
    'garment_chest_cm': garment_chest_cm,
    'fabric_stretch_pct': fabric_stretch_pct,
    'product_type_id': product_type_id,
    'fit_class': fit_class
})

print(f"Hybrid dataset created with {len(fitcheck_df)} real Kaggle records!")

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. One-Hot Encode product_type_id from the hybrid fitcheck_df
X_encoded = pd.get_dummies(
    fitcheck_df[['height_cm', 'weight_kg', 'imc_index', 'garment_chest_cm', 'fabric_stretch_pct', 'product_type_id']],
    columns=['product_type_id'],
    drop_first=False
)
y = fitcheck_df['fit_class']

# 2. Downsample for fast Grid Search execution (optional, use 15,000 rows for speed)
# With 119,404 rows, grid search on the full dataset can take 15+ minutes
X_sample, _, y_sample, _ = train_test_split(X_encoded, y, train_size=15000, random_state=42, stratify=y)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

# 4. Define and run Grid Search
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8]
}

print("Running Grid Search on hybrid dataset...")
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 5. Extract best model
best_gb_model = grid_search.best_estimator_
y_pred = best_gb_model.predict(X_test)

print(f"Optimized Accuracy: {accuracy_score(y_test, y_pred):.3f}")

# 6. Save model and feature artifacts into Models/ (the folder the API serves from)
joblib.dump(best_gb_model, MODEL_DIR / 'fitcheck_gb_model.joblib')
model_features = list(X_encoded.columns)
joblib.dump(model_features, MODEL_DIR / 'model_features.joblib')

print("Model and feature specs saved successfully!")

**1. Save the Model Artifact and Feature Columns**\
To deploy this as a microservice, the trained model and the feature column structure saved to ensure consistent one-hot encoding during live inference:

In [ ]:
# Save the trained model artifact (Models/ is mounted into the API container)
joblib.dump(best_gb_model, MODEL_DIR / 'fitcheck_gb_model.joblib')

# Save expected feature column names for live inference matching
model_features = list(X_encoded.columns)
joblib.dump(model_features, MODEL_DIR / 'model_features.joblib')

print("Model and feature specs saved successfully!")

**2. Creating an Inference Pipeline Helper** \
A lightweight prediction function that takes a raw user profile, applies the necessary transformations (derived BMI + One-Hot Encoding), and outputs a prediction that can be passed directly into our Gemini/MLflow prompt:

In [20]:
def predict_fit(user_data, model, feature_columns):
    """
    user_data dict structure:
    {
        'height_cm': 175.0,
        'weight_kg': 72.0,
        'garment_chest_cm': 98.0,
        'fabric_stretch_pct': 3.0,
        'product_type_id': 0  # 0: Tops, 1: Bottoms, 2: Jackets
    }
    """
    df = pd.DataFrame([user_data])

    # Calculate derived BMI
    df['imc_index'] = df['weight_kg'] / ((df['height_cm'] / 100.0) ** 2)

    # One-hot encode product type
    df_encoded = pd.get_dummies(df, columns=['product_type_id'])

    # Reindex to match training schema (fills missing dummy columns with 0)
    df_encoded = df_encoded.reindex(columns=feature_columns, fill_value=0)

    # Class mapping
    class_map = {0: "Too Small", 1: "Good Fit", 2: "Too Large"}
    prediction_id = model.predict(df_encoded)[0]
    probabilities = model.predict_proba(df_encoded)[0]

    return {
        "prediction_class": class_map[prediction_id],
        "confidence": float(probabilities[prediction_id]),
        "raw_class_id": int(prediction_id)
    }

# Quick Test Example:
sample_user = {
    'height_cm': 180.0,
    'weight_kg': 85.0,
    'garment_chest_cm': 92.0,  # Chest size might be small for an 85kg person
    'fabric_stretch_pct': 2.0,
    'product_type_id': 0
}

result = predict_fit(sample_user, best_gb_model, X_encoded.columns)
print("Sample Live Prediction:", result)

Sample Live Prediction: {'prediction_class': 'Too Small', 'confidence': 0.8542776212688131, 'raw_class_id': 0}


**Model Serialization & MLflow Tracking**\
Registering the model artifact and track the run using MLflow

In [ ]:
import mlflow
import mlflow.sklearn
import joblib
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score

# mlflow is installed in the image and the tracking URI + experiment were set in the setup cell,
# so this run shows up at http://localhost:5001 under EXPERIMENT_NAME.

# Ensure y_pred exists from best_gb_model
y_pred = best_gb_model.predict(X_test)

with mlflow.start_run(run_name="Gradient_Boosting_Hybrid"):
    # 1. Log Hyperparameters (+ how much data the model saw, useful when comparing runs)
    mlflow.log_params(best_gb_model.get_params())
    mlflow.log_param("train_rows", len(X_train))

    # 2. Log Performance Metrics
    #    f1_macro is logged too: classes are imbalanced, so accuracy alone hides errors on the
    #    smaller "Good Fit" / "Too Small" classes. Same metric names as the optimized run.
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))
    mlflow.log_metric("cv_accuracy", grid_search.best_score_)

    # 3. Log Model Artifact using cloudpickle serialization.
    #    signature + input_example document the expected (one-hot) input schema in the registry.
    mlflow.sklearn.log_model(
        sk_model=best_gb_model,
        name="model",
        serialization_format="cloudpickle",
        signature=infer_signature(X_test, y_pred),
        input_example=X_test.head(3),
        registered_model_name="FitCheck_GBClassifier"
    )

    # 4. Save local backup artifacts into Models/ (served by the API)
    joblib.dump(best_gb_model, MODEL_DIR / "fitcheck_gb_model.joblib")
    joblib.dump(list(X_encoded.columns), MODEL_DIR / "model_features.joblib")

print("Model tracked and registered in MLflow!")

**FastAPI service code**\
The API no longer lives in this notebook (the old `%%writefile app.py` cell created a second, drifting copy).
The single source of truth is `api/app.py`, which runs in the `fitcheck-api` container on http://localhost:3000.
It serves both the baseline and the optimized model plus the web UI. Feature engineering is shared with this notebook via `src/fitcheck/features.py`.

**Building and testing the FastAPI Microservice**

1. Check the model artifacts the API will serve (`Models/`)
2. Test the real `api/app.py` in-process with FastAPI's `TestClient` (the running container is at http://localhost:3000)

In [ ]:
# List the artifacts in Models/ (what the API container loads at startup)
for f in sorted(MODEL_DIR.glob("*.joblib")):
    print(f"{f.stat().st_size / 1024:10.1f} KB  {f.name}")

In [ ]:
import sys
import importlib
from fastapi.testclient import TestClient

# 1. Force Python to drop the cached API module so freshly saved models are re-loaded
sys.modules.pop("api.app", None)

# 2. Import the real service code from api/app.py (it loads models from MODEL_DIR on import)
api_module = importlib.import_module("api.app")

# 3. Re-initialize the TestClient with the updated app instance
client = TestClient(api_module.app)

# 4. Test Health Check Endpoint
health_res = client.get("/health")
print("Health Check:", health_res.status_code, health_res.json())

# 5. Test Prediction Endpoint on every model the API loaded (baseline, and optimized once trained)
payload = {
    "height_cm": 182.0,
    "weight_kg": 88.0,
    "garment_chest_cm": 90.0,
    "fabric_stretch_pct": 2.0,
    "product_type_id": 0  # Tops
}

for model_name in health_res.json()["available_models"]:
    pred_res = client.post("/predict", json={**payload, "model": model_name})
    print(f"[{model_name}] Prediction Response Status:", pred_res.status_code)
    print(f"[{model_name}] Prediction JSON:", pred_res.json())

---
## Optimized Model (`fitcheck_gb_model_optimized`)

The baseline above is kept **unchanged** so it can be compared fairly. The optimized model addresses what the review of the baseline found:

| Baseline issue | Optimized model |
| :--- | :--- |
| `GradientBoostingClassifier` is slow, so only **12,000 of 119,404 rows** (10%) were used for training | `HistGradientBoostingClassifier` (histogram-based, multi-threaded) trains on **all rows not held out** (~95k) |
| Trees split one feature at a time; the fit boundary is a *diagonal* (garment size **vs** body size), learned as a coarse staircase | Garment-relative-to-body features (`effective_chest_cm`, ratios and differences to height/weight) in `src/fitcheck/features.py` |
| Feature engineering (BMI, one-hot) is re-implemented in 3 places (training, `predict_fit`, API) → train/serve skew risk | A single sklearn **Pipeline** (features → model) that takes the raw API inputs. Native categorical support replaces one-hot |
| Selected by **accuracy** on imbalanced classes ("Too Large" is the clear majority) | Selected by **macro-F1** (each class counts equally), with `class_weight` in the search |
| Fixed number of trees (`n_estimators`) | Early stopping picks the number of boosting iterations |
| MLflow run without signature, input example or search results | Signature, input example, CV table, classification report, and the `src/fitcheck` code packaged with the model |

**Fair comparison:** both models are scored on the **same hold-out set** (20% of rows), which excludes every row the baseline was trained on and is never seen by the optimized model.

> ⚠️ **Important caveat about the data:** `fit_class` is produced by a known rule (`garment + 0.5·stretch − (0.45·weight + 0.35·height)` vs ±4 cm). Both models can only learn to reproduce *that rule*, so accuracy here measures how well they imitate the heuristic, not real-world fit. `product_type_id` has no effect on the label at all. Validating against real return/fit feedback is the next step before trusting the numbers.

Run the next two cells (training takes a few minutes), then open MLflow at http://localhost:5001 to compare runs, and restart the API (`docker compose restart api`) to serve the new model.

In [ ]:
import time
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Shared with the API (src/fitcheck/features.py) so training and serving use identical code
from fitcheck.features import CLASS_LABELS, RAW_FEATURES, add_fit_features

# ==========================================
# 1. SHARED HOLD-OUT SET (never seen by either model)
# ==========================================
y_all = fitcheck_df["fit_class"]

# Re-derive exactly which rows the BASELINE trained on, using the same calls and seeds as the
# baseline cell. train_test_split selects rows from (n_rows, y, random_state) only, so splitting
# the index reproduces the baseline's selection even if that cell was not run in this session.
baseline_sample_idx, _ = train_test_split(fitcheck_df.index, train_size=15000, random_state=42, stratify=y_all)
baseline_train_idx, _ = train_test_split(baseline_sample_idx, test_size=0.2, random_state=42)
if "X_train" in globals():  # sanity check against the baseline cell, when it was run
    assert set(X_train.index) == set(baseline_train_idx), "Baseline split could not be reproduced"

# Hold-out = 20% of all rows, drawn only from rows the baseline never trained on
candidate_idx = fitcheck_df.index.difference(baseline_train_idx)
_, holdout_idx = train_test_split(
    candidate_idx, test_size=int(0.2 * len(fitcheck_df)), random_state=42, stratify=y_all.loc[candidate_idx]
)
# The optimized model can train on everything else (including the baseline's training rows)
opt_train_idx = fitcheck_df.index.difference(holdout_idx)

# The pipeline takes RAW inputs (same fields the API receives); features are built inside it
X_opt_train, y_opt_train = fitcheck_df.loc[opt_train_idx, RAW_FEATURES], y_all.loc[opt_train_idx]
X_holdout, y_holdout = fitcheck_df.loc[holdout_idx, RAW_FEATURES], y_all.loc[holdout_idx]
print(f"Optimized train rows: {len(X_opt_train):,} | shared hold-out rows: {len(X_holdout):,}")

# ==========================================
# 2. PIPELINE: feature engineering -> histogram gradient boosting
# ==========================================
optimized_pipeline = Pipeline([
    # BMI + garment-vs-body features; a module function so the pickled pipeline loads in the API
    ("features", FunctionTransformer(add_fit_features)),
    ("model", HistGradientBoostingClassifier(
        categorical_features=["product_type_id"],  # native categorical handling (no one-hot)
        max_iter=1000,                             # upper bound; early stopping picks the real number
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42,
    )),
])

# ==========================================
# 3. RANDOMIZED HYPERPARAMETER SEARCH (selected by macro-F1)
# ==========================================
param_distributions = {
    "model__learning_rate": loguniform(0.03, 0.3),
    "model__max_leaf_nodes": randint(15, 96),
    "model__min_samples_leaf": randint(10, 80),
    "model__l2_regularization": loguniform(1e-3, 1.0),
    "model__class_weight": [None, "balanced"],  # "balanced" up-weights the smaller classes
}
N_ITER, CV_FOLDS = 15, 3

search = RandomizedSearchCV(
    estimator=optimized_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42),
    n_jobs=1,  # HistGradientBoosting already uses all CPU cores; parallel CV would oversubscribe
    random_state=42,
    verbose=1,
)

print("Running randomized search for the optimized model...")
t0 = time.perf_counter()
search.fit(X_opt_train, y_opt_train)
train_seconds = time.perf_counter() - t0
optimized_model = search.best_estimator_

# ==========================================
# 4. EVALUATE ON THE SHARED HOLD-OUT
# ==========================================
y_holdout_pred = optimized_model.predict(X_holdout)
opt_metrics = {
    "accuracy": accuracy_score(y_holdout, y_holdout_pred),
    "f1_macro": f1_score(y_holdout, y_holdout_pred, average="macro"),
    "balanced_accuracy": balanced_accuracy_score(y_holdout, y_holdout_pred),
    "cv_f1_macro": search.best_score_,
    "search_seconds": train_seconds,
}
report = classification_report(y_holdout, y_holdout_pred, target_names=list(CLASS_LABELS.values()), digits=4)
print(f"Search finished in {train_seconds:.0f}s | best params: {search.best_params_}")
print(f"Hold-out accuracy: {opt_metrics['accuracy']:.4f} | macro-F1: {opt_metrics['f1_macro']:.4f}")
print(report)

# ==========================================
# 5. SAVE FOR THE API + TRACK/REGISTER IN MLFLOW
# ==========================================
joblib.dump(optimized_model, MODEL_DIR / "fitcheck_gb_model_optimized.joblib")

# Search results table (one row per sampled hyperparameter set), best first
cv_table = pd.DataFrame(search.cv_results_)
cv_table = cv_table[
    [c for c in cv_table.columns if c.startswith("param_")]
    + ["mean_test_score", "std_test_score", "rank_test_score", "mean_fit_time"]
].sort_values("rank_test_score")

with mlflow.start_run(run_name="HistGB_Pipeline_optimized"):
    # Hyperparameters (strip the "model__" pipeline prefix for readability in the UI)
    mlflow.log_params({k.replace("model__", ""): v for k, v in search.best_params_.items()})
    mlflow.log_params({
        "model_type": "Pipeline(add_fit_features -> HistGradientBoostingClassifier)",
        "train_rows": len(X_opt_train),
        "holdout_rows": len(X_holdout),
        "search_iterations": N_ITER,
        "cv_folds": CV_FOLDS,
        "boosting_iterations": optimized_model.named_steps["model"].n_iter_,  # chosen by early stopping
    })
    mlflow.log_metrics(opt_metrics)

    # Extra artifacts: full search table + per-class report
    mlflow.log_text(cv_table.to_csv(index=False), "search/cv_results.csv")
    mlflow.log_dict(
        classification_report(y_holdout, y_holdout_pred, target_names=list(CLASS_LABELS.values()), output_dict=True),
        "evaluation/classification_report.json",
    )

    # Register the whole pipeline. It accepts the raw API fields, and code_paths ships
    # src/fitcheck with the model so it can be loaded outside this project too.
    mlflow.sklearn.log_model(
        sk_model=optimized_model,
        name="model",
        serialization_format="cloudpickle",
        signature=infer_signature(X_holdout.head(200), optimized_model.predict(X_holdout.head(200))),
        input_example=X_holdout.head(3),
        code_paths=[str(SRC_DIR / "fitcheck")],
        registered_model_name="FitCheck_GBClassifier_optimized",
    )

print("Optimized model saved to Models/ and registered in MLflow as 'FitCheck_GBClassifier_optimized'!")

In [ ]:
import time
import joblib
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, log_loss, recall_score)

from fitcheck.features import CLASS_LABELS, build_baseline_features

# ==========================================
# BASELINE vs OPTIMIZED on the same hold-out set
# ==========================================
# Load both models from Models/, i.e. exactly the files the API serves
# (your baseline may be the one trained in Colab; it used the same seed, so the same rows).
baseline_model = joblib.load(MODEL_DIR / "fitcheck_gb_model.joblib")
baseline_columns = joblib.load(MODEL_DIR / "model_features.joblib")
optimized_model = joblib.load(MODEL_DIR / "fitcheck_gb_model_optimized.joblib")

# Each model gets the input format it was trained on
model_inputs = {
    "baseline": (baseline_model, build_baseline_features(X_holdout, baseline_columns)),  # one-hot schema
    "optimized": (optimized_model, X_holdout),                                            # raw fields
}

y_true = y_holdout.to_numpy()
labels = list(CLASS_LABELS)                       # [0, 1, 2]
label_names = list(CLASS_LABELS.values())         # ["Too Small", "Good Fit", "Too Large"]

# "Hard cases": rows whose true chest clearance is within 2 cm of a size boundary (±4 cm).
# This is where sizing mistakes actually happen, so it separates the models better than overall accuracy.
holdout_delta = chest_delta[np.asarray(holdout_idx)]
near_boundary = np.abs(np.abs(holdout_delta) - 4.0) < 2.0

rows, confusion = [], {}
for name, (model, X_in) in model_inputs.items():
    t0 = time.perf_counter()
    proba = model.predict_proba(X_in)
    elapsed = time.perf_counter() - t0
    pred = model.classes_[proba.argmax(axis=1)]
    recalls = recall_score(y_true, pred, labels=labels, average=None)

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "f1_macro": f1_score(y_true, pred, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, proba, labels=labels),        # lower = better-calibrated confidence
        "accuracy_near_boundary": accuracy_score(y_true[near_boundary], pred[near_boundary]),
        "recall_too_small": recalls[0],
        "recall_good_fit": recalls[1],
        "recall_too_large": recalls[2],
        "predict_ms_per_1k_rows": elapsed / len(X_in) * 1e6,
    })
    confusion[name] = pd.DataFrame(
        confusion_matrix(y_true, pred, labels=labels),
        index=[f"true: {n}" for n in label_names],
        columns=[f"pred: {n}" for n in label_names],
    )

# Side-by-side table (+ difference column: positive = optimized is higher)
comparison = pd.DataFrame(rows).set_index("model").T
comparison["optimized - baseline"] = comparison["optimized"] - comparison["baseline"]
print(f"Shared hold-out: {len(y_true):,} rows ({near_boundary.sum():,} near a size boundary)")
display(comparison.style.format("{:.4f}"))

for name, cm in confusion.items():
    print(f"\nConfusion matrix: {name}")
    display(cm)

# Log the comparison as its own MLflow run so it sits next to both training runs in the UI
with mlflow.start_run(run_name="Comparison_baseline_vs_optimized"):
    mlflow.log_param("holdout_rows", len(y_true))
    mlflow.log_param("near_boundary_rows", int(near_boundary.sum()))
    for row in rows:
        mlflow.log_metrics({f"{row['model']}_{k}": v for k, v in row.items() if k != "model"})
    mlflow.log_text(comparison.to_csv(), "comparison/metrics.csv")

print("Comparison logged to MLflow (run 'Comparison_baseline_vs_optimized').")